## GPT2 implementation


KV cache


In [ ]:
class Cache:

	def __init__(self, n, max_seq_len, h)

Normalization techniques

- batchnorm
- layernorm
- RMSnorm
- pre- or post-LayerNorm


In [ ]:
class RMSNorm(nn.Module): 

	def __init__(self, d_embed:int, eps:float=1e-4): 
		
		super().__init__()
		self.eps = eps
		
		self.weight = nn.Parameter(d_embed)

	def forward(self, x_in: torch.Tensor): 

		variance = x_in.pow(2).mean(dim=-1, keepdim=True) # mean sum of squares along feature dimension, (B, T, D)

		x_rms = torch.rsqrt(variance + self.eps) # (B, T, D)

		return x_in * x_rms * self.weight

### NOTE: pre- and post-LN are just whether you place it before the layer, or after the layer + residual stream
class LayerNorm(nn.Module):

	def __init__(self, d_embed: int, eps:float=1e-5):

		super().__init__()

		self.eps = eps
		self.weight = nn.Parameter(torch.ones(d_embed))
		self.bias = nn.Parameter(torch.zeros(d_embed))

	def forward(self, x_in):

		b, t, d = x_in.shape
		
		variance = x_in.var(dim=-1, keepdim=True) # not using Bessel's correction
		mean = x_in.mean(dim=-1, keepdim=True)

		x_out = (x_in - mean) * torch.rsqrt(variance + self.eps) # reciprocal square root

		return x_out * self.weight + self.bias

class BatchNorm(nn.Module):

	def __init__(self, d_embed:int, eps:float=1e-5, momentum:float=0.1): 
		
		super().__init__()

		self.eps = eps
		self.d_embed = d_embed
		self.momentum = momentum

		self.weight = nn.Parameter(torch.ones(d_embed))
		self.bias = nn.Parameter(torch.zeros(d_embed))

		self.register_buffer('running_mean', torch.zeros(d_embed))
		self.register_buffer('running_var', torch.zeros(d_embed))

	def forward(self, x_in:torch.Tensor, training:bool=False): 

		b, t, d = x_in.shape
		x_in = x_in.flatten(0,1)
		
		if training: 
			
			variance = x_in.var(dim=0, keepdim=True, unbiased=True) # (b*t, d)
			mean = x_in.mean(dim=0, keepdim=True)

			with torch.no_grad():
				self.running_mean.mul_(1-self.momentum).add_(self.momentum*mean)
				self.running_var.mul_(1-self.momentum).add_(self.momentum*x_in.var(dim=0, keepdim=True, unbiased=False)) # since here we are actually estimating, we use Bessel's

		else: 

			mean, variance = self.running_mean, self.running_var
		
		x_out = (x_in - mean) * torch.rsqrt(variance + self.eps)
		x_out = x_out.view(b, t, -1)

		return x_out * self.weight + self.bias



Residual connections

- implementing a residual stream
- implementing ByteDance HC
- implementing Deepseek mHC


Learning rate scheduler

- cosine decay


Optimizer

- AdamW
- MUON
- SHAMPOO


Dropout

_dropout as ensemble learning_

- dropout acts as ensemble learning during inference. During training time, say we set dropout = 0.99, thus we train lots of different sub-networks, each lighting up a different set of neurons. Each of these neurons learns to produce the output, independent of each other, or if there's neurons shared between subnetworks, the neuron becomes a shared parameter, but this weakens the ensemble learning since if there's a lot of shared neurons, your subnetworks are not producing meaningfully different outputs.
- at inference time, we use .eval() to turn off dropout. Now each subnetwork is activated in the forward pass, and the activations are a combination of all the subnetworks votes on what the right activation is, since each has learned a different 'correct answer' for the output vector.

_dropout during self-attention_

- we place dropout in 2 different places
  - after softmax produces the scores
  - right before we merge with the layer output with the residual stream
- why there?
  - we don't want to dropout the residual stream, rather we care about the weights in the layer, learning to develop subnetworks. The residual stream changes on every token - this is not something the model needs to learn

  - models can become overly reliant and attend only to the tokens immediately adjacent to the next token (e.g. the cat chases the dog. It was very fast. It refers to the \_**\_ -> the model may learn, when predicting the \_\_** to attend only to the closest tokens, so 'dog' when 'it' may refer to the cat)


Implement GPT2 from scratch


In [71]:
import torch
from collections import defaultdict
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset
from torch.utils.data import TensorDataset

# tokenize the data
class Tokenizer:

	def __init__(self): 
		
		# initialize a dictionary that is the 256 unicode characters
		self.vocab = {idx: bytes([idx]) for idx in range(256)} # bytes([4]) creates a bytes object for the value 4, while bytes(4) creates a bytes object of length 4 - initialized with zeros
		self.merges = defaultdict(int)
	
	def get_stats(self, ids):

		# given a sequence find the most common occuring pairs
		counts = defaultdict(int)
		for pair in zip(ids, ids[1:]): 
			counts[pair] += 1 # initializes a key of [id1, id2] and then adds to the count

		return counts

	def merge_seq(self, ids, pair, idx):

		new_ids = []
		i = 0

		while i < len(ids):
			if i < len(ids)-1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
				i += 2
				new_ids.append(idx)
			else: 
				new_ids.append(ids[i])
				i += 1

		return new_ids

	def merge(self, ids, num_merges): 

		# now we iterate through all of the token ids, and merge 

		for i in range(num_merges): 

			# find the most common occuring pair
			stats = self.get_stats(ids)
			freq_pair = max(stats, key=stats.get) 

			# with the most frequent pair, merge the sequence and add it to the merges list
			token_id = 256 + i
			ids = self.merge_seq(ids, freq_pair, token_id) # newly merged sequence
			self.merges[freq_pair] = token_id # add this to the merges dictionary 

		for (p0, p1), idx in self.merges.items(): 

			self.vocab[idx] = self.vocab[p0] + self.vocab[p1] # p0,p1 = 145, 165, the new vocab idx = 257, so now vocab[257] = bytes(145) + bytes(165) (concat - since we indexed into vocab,s grabbed the bytes and concatenated)

	def encode(self, text, num_merges=50000): 

		ids = text.encode('utf-8')

		merge(self, ids, num_merges)

		# now run it through the vocab dict
		for pair, idx in self.merges.items():
			# for i in range(len(merges)): 
			ids = self.merge_seq(ids, pair, idx) # given the amount of merges, perform all of them sequentially

		return ids

	def decode(self, ids):

		new_ids = [self.vocab[idx] for idx in ids]
		tokens = b"".join(new_ids)
		text = tokens.decode('utf-8', errors='replace')
		
		return text

tk = Tokenizer()

**Dataloader**

- dataloader that loads from external url (online dataset / hf)
- dataloader that takes in ids and converts to (B, T, D)


In [ ]:
import requests
import os

cdir = os.path.abspath('')
data_path = os.path.join(cdir, "data")

os.makedirs(data_path, exist_ok=True) # make directory if doesn't exist
input_file_path = os.path.join(data_path, "input.txt")

# input_file_path = os.path.join(os.path.dirname(__file__), 'input.txt') # only works in .py files

if not os.path.exists(input_file_path): # if nothing exists at the file path right now
	data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
	with open(input_file_path, 'w') as f: # open at the input file path, with write permissions - creating a file since it doesn't exist, naming it as f
		f.write(requests.get(data_url).text)

with open(input_file_path, 'r') as f:
	data = f.read()

data = tk.encode(data)

In [ ]:
import torch
import numpy as np

ids = torch.tensor(data, dtype=torch.long)

training_split = int(len(ids) * 0.9)

train_data, test_data = ids[:training_split], ids[training_split:] # training_split = 0.9

train_ids = np.asarray(train_data, dtype=np.uint16) # does not copy in memory
test_ids = np.asarray(test_data, dtype=np.uint16)

train_ids.tofile(os.path.join(data_path, 'train.bin'))
test_ids.tofile(os.path.join(data_path, 'test.bin'))

def get_batch(split, batch_size=64, max_seq_len=1024):
	d = train_data if split == 'train' else test_data

	# index into random integers into the train_data, do this along the batch_size dimension
	# we want any integers before (lenght of the data - longest sequence)
	ix = torch.randint(len(d)-max_seq_len, (batch_size,)) # (batch_size,)
	x = torch.stack([d[i: i+max_seq_len] for i in ix])
	y = torch.stack([d[i+1: i+max_seq_len+1] for i in ix])
	return x,y # train, targets

train_data, _ = get_batch('train')